# Κεφάλαιο 7: Δημιουργία Εφαρμογών Συνομιλίας
## Γρήγορη Εκκίνηση OpenAI API

Αυτό το σημειωματάριο είναι προσαρμοσμένο από το [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) που περιλαμβάνει σημειωματάρια που έχουν πρόσβαση στις υπηρεσίες [Azure OpenAI](notebook-azure-openai.ipynb).

Το Python OpenAI API λειτουργεί και με τα Azure OpenAI Models, με κάποιες τροποποιήσεις. Μάθετε περισσότερα για τις διαφορές εδώ: [Πώς να αλλάξετε μεταξύ των OpenAI και Azure OpenAI endpoints με Python](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Επισκόπηση  
"Τα μεγάλα γλωσσικά μοντέλα είναι συναρτήσεις που απεικονίζουν κείμενο σε κείμενο. Δεδομένης μιας εισόδου κειμένου, ένα μεγάλο γλωσσικό μοντέλο προσπαθεί να προβλέψει το κείμενο που θα ακολουθήσει"(1). Αυτό το σημειωματάριο "γρήγορης εκκίνησης" θα εισαγάγει τους χρήστες σε βασικές έννοιες των LLM, βασικές απαιτήσεις πακέτων για να ξεκινήσουν με το AML, μια ήπια εισαγωγή στο σχεδιασμό προτροπών, και αρκετά σύντομα παραδείγματα διαφορετικών περιπτώσεων χρήσης. 


## Πίνακας Περιεχομένων  

[Επισκόπηση](#overview)  
[Πώς να χρησιμοποιήσετε την Υπηρεσία OpenAI](#how-to-use-openai-service)  
[1. Δημιουργία της Υπηρεσίας OpenAI](#1.-creating-your-openai-service)  
[2. Εγκατάσταση](#2.-installation)    
[3. Πιστοποιητικά](#3.-credentials)  

[Περιπτώσεις Χρήσης](#use-cases)    
[1. Περίληψη Κειμένου](#1.-summarize-text)  
[2. Ταξινόμηση Κειμένου](#2.-classify-text)  
[3. Δημιουργία Νέων Ονομάτων Προϊόντων](#3.-generate-new-product-names)  
[4. Λεπτομερής Ρύθμιση Ταξινομητή](#4.fine-tune-a-classifier)  

[Αναφορές](#references)


### Δημιουργήστε το πρώτο σας prompt  
Αυτή η σύντομη άσκηση θα παρέχει μια βασική εισαγωγή για την υποβολή prompts σε ένα μοντέλο OpenAI για μια απλή εργασία "περίληψη".


**Βήματα**:  
1. Εγκαταστήστε τη βιβλιοθήκη OpenAI στο περιβάλλον python σας  
2. Φορτώστε τις τυπικές βοηθητικές βιβλιοθήκες και ορίστε τα τυπικά διαπιστευτήρια ασφαλείας OpenAI για την Υπηρεσία OpenAI που έχετε δημιουργήσει  
3. Επιλέξτε ένα μοντέλο για την εργασία σας  
4. Δημιουργήστε ένα απλό prompt για το μοντέλο  
5. Υποβάλετε το αίτημά σας στο API του μοντέλου!


### 1. Εγκατάσταση OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Εισαγωγή βοηθητικών βιβλιοθηκών και δημιουργία διαπιστευτηρίων


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Βρίσκοντας το σωστό μοντέλο  
Μοντέλα όπως τα GPT-4o και GPT-4o mini μπορούν να κατανοήσουν και να δημιουργήσουν φυσική γλώσσα, και είναι διαθέσιμα στην πλατφόρμα OpenAI με διαφορετικά επίπεδα ισχύος και ταχύτητας κατάλληλα για διαφορετικές εργασίες.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. Σχεδιασμός Προτροπών  

"Η μαγεία των μεγάλων γλωσσικών μοντέλων έγκειται στο ότι, εκπαιδευόμενα να ελαχιστοποιούν αυτό το σφάλμα πρόβλεψης σε τεράστιες ποσότητες κειμένου, τα μοντέλα τελικά μαθαίνουν έννοιες χρήσιμες για αυτές τις προβλέψεις. Για παράδειγμα, μαθαίνουν έννοιες όπως"(1):

* πώς να ορθογραφούν
* πώς λειτουργεί η γραμματική
* πώς να παραφράζουν
* πώς να απαντούν σε ερωτήσεις
* πώς να διατηρούν μια συνομιλία
* πώς να γράφουν σε πολλές γλώσσες
* πώς να προγραμματίζουν
* κ.λπ.

#### Πώς να ελέγχετε ένα μεγάλο γλωσσικό μοντέλο  
"Από όλα τα εισερχόμενα σε ένα μεγάλο γλωσσικό μοντέλο, με διαφορά το πιο επιδραστικό είναι η προτροπή κειμένου(1).

Τα μεγάλα γλωσσικά μοντέλα μπορούν να κινητοποιηθούν να παράγουν έξοδο με μερικούς τρόπους:

Οδηγία: Πείτε στο μοντέλο τι θέλετε
Ολοκλήρωση: Προκαλέστε το μοντέλο να ολοκληρώσει την αρχή αυτού που θέλετε
Επίδειξη: Δείξτε στο μοντέλο τι θέλετε, με:
Λίγα παραδείγματα στην προτροπή
Πολλά εκατοντάδες ή χιλιάδες παραδείγματα σε ένα εκπαιδευτικό σύνολο για βελτιστοποίηση"



#### Υπάρχουν τρεις βασικές οδηγίες για τη δημιουργία προτροπών:

**Δείξτε και πείτε**. Κάντε σαφές τι θέλετε είτε μέσω οδηγιών, παραδειγμάτων ή ενός συνδυασμού και των δύο. Αν θέλετε το μοντέλο να ταξινομήσει μια λίστα αντικειμένων με αλφαβητική σειρά ή να κατηγοριοποιήσει μια παράγραφο με βάση το συναίσθημα, δείξτε του ότι αυτό θέλετε.

**Παρέχετε ποιοτικά δεδομένα**. Αν προσπαθείτε να φτιάξετε έναν ταξινομητή ή να κάνετε το μοντέλο να ακολουθήσει ένα μοτίβο, βεβαιωθείτε ότι υπάρχουν αρκετά παραδείγματα. Φροντίστε να διορθώσετε τα παραδείγματά σας — το μοντέλο είναι συνήθως έξυπνο αρκετά για να διακρίνει βασικά ορθογραφικά λάθη και να σας δώσει απάντηση, αλλά μπορεί επίσης να υποθέσει ότι αυτό είναι σκόπιμο και μπορεί να επηρεάσει την απάντηση.

**Ελέγξτε τις ρυθμίσεις σας.** Οι ρυθμίσεις temperature και top_p ελέγχουν πόσο ντετερμινιστικό είναι το μοντέλο στη δημιουργία της απάντησης. Αν ζητάτε από αυτό μια απάντηση για την οποία υπάρχει μόνο μια σωστή λύση, τότε θα θέλατε να τις ορίσετε χαμηλότερα. Αν ψάχνετε για πιο ποικίλες απαντήσεις, τότε μπορεί να θέλετε να τις ορίσετε πιο ψηλά. Το πιο κοινό λάθος που κάνουν οι άνθρωποι με αυτές τις ρυθμίσεις είναι ότι υποθέτουν πως είναι ρυθμίσεις "ευφυΐας" ή "δημιουργικότητας".


Πηγή: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Υποβολή!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Επαναλάβετε την ίδια κλήση, πώς συγκρίνονται τα αποτελέσματα;


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Περίληψη Κειμένου  
#### Πρόκληση  
Περίληψη κειμένου προσθέτοντας ένα 'tl;dr:' στο τέλος ενός αποσπάσματος κειμένου. Παρατηρήστε πώς το μοντέλο καταλαβαίνει πώς να εκτελεί έναν αριθμό εργασιών χωρίς επιπλέον οδηγίες. Μπορείτε να πειραματιστείτε με πιο περιγραφικές προτροπές από το tl;dr για να τροποποιήσετε τη συμπεριφορά του μοντέλου και να προσαρμόσετε την περίληψη που λαμβάνετε(3).  

Πρόσφατες εργασίες έχουν αποδείξει σημαντικά κέρδη σε πολλές εργασίες και προκλήσεις στη Επεξεργασία Φυσικής Γλώσσας (NLP) με προεκπαίδευση σε μεγάλο σώμα κειμένων και στη συνέχεια λεπτομερή εκπαίδευση σε συγκεκριμένη εργασία. Ενώ είναι συνήθως ανεξάρτητη από την εργασία στην αρχιτεκτονική, αυτή η μέθοδος απαιτεί ακόμα σύνολα δεδομένων εκπαίδευσης προσανατολισμένα σε εργασία με χιλιάδες ή δεκάδες χιλιάδες παραδείγματα. Αντίθετα, οι άνθρωποι μπορούν γενικά να εκτελέσουν μια νέα γλωσσική εργασία με μόνο λίγα παραδείγματα ή με απλές οδηγίες - κάτι που τα τρέχοντα συστήματα NLP δυσκολεύονται ακόμα σε μεγάλο βαθμό να κάνουν. Εδώ αποδεικνύουμε ότι η κλιμάκωση των γλωσσικών μοντέλων βελτιώνει σημαντικά την απόδοση σε ανεξάρτητη από εργασία εκτέλεση με λίγα παραδείγματα, μερικές φορές φτάνοντας ακόμα και σε ανταγωνιστικότητα με προηγούμενες μεθόδους λεπτομερούς εκπαίδευσης που θεωρούνται κορυφαίες. 



Tl;dr


# Ασκήσεις για διάφορες περιπτώσεις χρήσης  
1. Σύνοψη Κειμένου  
2. Ταξινόμηση Κειμένου  
3. Δημιουργία Νέων Ονομάτων Προϊόντων


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Ταξινόμηση Κειμένου  
#### Πρόκληση  
Ταξινομήστε αντικείμενα σε κατηγορίες που παρέχονται κατά την εκτέλεση. Στο παρακάτω παράδειγμα, παρέχουμε τόσο τις κατηγορίες όσο και το κείμενο προς ταξινόμηση στο prompt(*playground_reference). 

Ερώτηση Πελάτη: Γεια σας, ένα από τα πλήκτρα στο πληκτρολόγιο του laptop μου χάλασε πρόσφατα και θα χρειαστώ αντικατάσταση:

Ταξινομημένη κατηγορία:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Δημιουργία Νέων Ονομάτων Προϊόντων
#### Πρόκληση
Δημιουργήστε ονόματα προϊόντων από παραδείγματα λέξεων. Εδώ περιλαμβάνουμε στην υπόδειξη πληροφορίες σχετικά με το προϊόν για το οποίο πρόκειται να δημιουργήσουμε ονόματα. Παρέχουμε επίσης ένα παρόμοιο παράδειγμα για να δείξουμε το μοτίβο που επιθυμούμε να λάβουμε. Έχουμε ορίσει επίσης υψηλή την τιμή θερμοκρασίας για να αυξήσουμε τον τυχαίο χαρακτήρα και τις πιο καινοτόμες απαντήσεις.

Περιγραφή προϊόντος: Ένα μηχάνημα για milkshake στο σπίτι
Λέξεις-κλειδιά: γρήγορο, υγιεινό, συμπαγές.
Ονόματα προϊόντων: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Περιγραφή προϊόντος: Ένα ζευγάρι παπούτσια που μπορεί να ταιριάξει σε οποιοδήποτε μέγεθος ποδιού.
Λέξεις-κλειδιά: προσαρμόσιμο, ταιριαστό, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Αναφορές  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Καλές πρακτικές για τη λεπτομερή ρύθμιση των μοντέλων GPT για ταξινόμηση κειμένου](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Για Περισσότερη Βοήθεια  
[Ομάδα Εμπορευματοποίησης OpenAI](AzureOpenAITeam@microsoft.com) 


# Συνεισφέροντες
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
